# 04 · Validate — the novelty-vs-scRMSD frontier + tool comparison

**Standard slot:** *validate (in silico).* **For Project 03 this is the core science:** the
**novelty-vs-foldability Pareto frontier**, per-topology and per-length success rates, and an honest
RFdiffusion / FrameFlow / Genie2 comparison scaffold (D3 part 2).

Needs `results/backbones.csv` from nb 02.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The novelty-vs-scRMSD frontier

Scatter **novelty (TM-score to PDB, lower = more novel)** against **scRMSD (lower = more foldable)**,
colored by topology. The horizontal line is the foldability bar (scRMSD < 2 Å); the vertical line is
the novelty threshold (TM < 0.5). The interesting designs are the **lower-left** quadrant: foldable
*and* novel. The **Pareto frontier** is the envelope no other design beats on both axes.

> Any scatter here is labeled **EXAMPLE_DATA** when run on the mock backend — synthetic points, never
> presented as real designs. Re-run on real RFdiffusion output for your thesis figure.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import rfdiff_tools as rt

bb = pd.read_csv("results/backbones.csv")
synthetic = bool(bb.get("synthetic", pd.Series([False])).any())
tag = "EXAMPLE_DATA (synthetic)" if synthetic else "real designs"

colors = {"alpha": "tab:red", "beta": "tab:blue", "mixed": "tab:green"}
fig, ax = plt.subplots(figsize=(6, 5))
for ss, g in bb.groupby("ss_bias"):
    ax.scatter(g["tm_to_pdb"], g["scrmsd"], s=18 + (g["length"] - 80) / 4,
               c=colors.get(ss, "gray"), alpha=0.7, label=ss, edgecolors="none")
ax.axhline(rt.SELF_CONSISTENT_SCRMSD, ls="--", c="k", lw=0.8)
ax.axvline(rt.NOVEL_TM, ls=":", c="gray", lw=0.8)
ax.set_xlabel("TM-score to nearest PDB fold  (lower = more novel)")
ax.set_ylabel("scRMSD (A)  (lower = more foldable)")
ax.set_title(f"Novelty vs foldability — {tag}\n(marker size ~ length; dashed = scRMSD<2; dotted = TM<0.5)")
ax.legend(title="topology")
plt.tight_layout(); plt.savefig("results/frontier.png", dpi=150); plt.show()
print("foldable & novel (lower-left):",
      int(((bb.scrmsd < rt.SELF_CONSISTENT_SCRMSD) & (bb.tm_to_pdb < rt.NOVEL_TM)).sum()), "/", len(bb))

## 2 · The Pareto frontier explicitly

Extract the non-dominated set: a design is on the frontier if no other design is both more novel
(lower TM) **and** more foldable (lower scRMSD). Reading the lowest TM achievable at scRMSD < 2 Å off
this set gives the raw material for the **novelty budget** (notebook 05).

In [ ]:
def pareto_front(df, x="tm_to_pdb", y="scrmsd"):
    """Non-dominated set minimizing BOTH x (TM, want low=novel) and y (scRMSD, want low=foldable)."""
    pts = df[[x, y]].dropna().sort_values([x, y]).values
    front, best_y = [], np.inf
    for xi, yi in pts:
        if yi < best_y:                 # strictly better foldability as we move to higher novelty
            front.append((xi, yi)); best_y = yi
    return pd.DataFrame(front, columns=[x, y])

front = pareto_front(bb)
print("Pareto frontier points (TM, scRMSD):")
print(front.round(3).to_string(index=False))
foldable_front = front[front["scrmsd"] < rt.SELF_CONSISTENT_SCRMSD]
if len(foldable_front):
    print(f"\nMost novel still foldable: TM={foldable_front['tm_to_pdb'].min():.3f} "
          f"at scRMSD={foldable_front.loc[foldable_front['tm_to_pdb'].idxmin(),'scrmsd']:.2f} A")

## 3 · Per-topology and per-length success rates

The honest hit-rate table: fraction with scRMSD < 2 Å in each (topology × length) cell. Expect the
characteristic pattern — high for short all-α, collapsing toward long all-β. Caption every figure
with the metric, cutoff, and N.

In [ ]:
bb["foldable"] = bb["scrmsd"] < rt.SELF_CONSISTENT_SCRMSD
rate = bb.groupby(["ss_bias", "length"])["foldable"].agg(["mean", "count"])
print("Foldable rate (scRMSD < 2 A) by topology x length  [", tag, "]:")
print(rate.assign(mean=rate["mean"].round(2)))

pivot = bb.groupby(["ss_bias", "length"])["foldable"].mean().unstack("length")
fig, ax = plt.subplots(figsize=(5.5, 3))
im = ax.imshow(pivot.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel("length"); ax.set_ylabel("topology")
ax.set_title(f"Foldable rate (scRMSD<2 A) — {tag}")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, "" if np.isnan(v) else f"{v:.2f}", ha="center", va="center",
                color="w" if (np.isnan(v) or v < 0.5) else "k", fontsize=9)
fig.colorbar(im, ax=ax, label="foldable rate")
plt.tight_layout(); plt.savefig("results/success_heatmap.png", dpi=150); plt.show()

## 4 · Tool comparison scaffold — RFdiffusion / FrameFlow / Genie2 `[extension]`

Compare generators on **diversity, speed, and foldability**. This is a *scaffold*: run each tool on
the same grid and fill the table from **your own measurements**. **Do NOT fabricate numbers** — the
cells below are `None`/`NaN` until you measure them. The mock backend cannot stand in for a real
generator comparison.

In [ ]:
import pandas as pd
# Fill ONLY from your own runs. Leave NaN until measured — never invent a generator's numbers.
comparison = pd.DataFrame({
    "tool":            ["RFdiffusion", "FrameFlow", "Genie2"],
    "foldable_rate":   [None, None, None],   # frac scRMSD<2 on the SAME grid (you measure)
    "median_tm_to_pdb":[None, None, None],   # novelty (you measure)
    "diversity":       [None, None, None],   # e.g. mean pairwise TM among designs (you measure)
    "sec_per_backbone":[None, None, None],   # wall-clock on YOUR GPU (you measure)
})
comparison.to_csv("results/tool_comparison.csv", index=False)
print("Tool-comparison scaffold (UNMEASURED — fill from your own runs, do not fabricate):")
print(comparison.to_string(index=False))

## D3 (part 2) checklist
- [ ] Novelty-vs-scRMSD frontier figure (lower-left = novel & foldable), with N and cutoffs captioned.
- [ ] Explicit Pareto frontier + the most-novel-still-foldable point identified.
- [ ] Per-topology AND per-length success-rate table/heatmap (honest, including the failure cells).
- [ ] Tool-comparison scaffold filled from **your own** RFdiffusion/FrameFlow/Genie2 runs (or left blank, not fabricated).
- [ ] Failure-mode notes: where does foldability collapse, and why (length / β-register)?

**Next:** `05_validation_plan.ipynb` — novelty budget + synthesis plan with paired controls.